In [1]:
# Mounting the google drive
from google.colab import drive
drive.mount('/content/gdrive')

ModuleNotFoundError: No module named 'google.colab'

In [7]:
import tensorflow as tf

In [2]:
# Google Drive Path
gDrivePath = "gdrive/MyDrive/Colab Notebooks/COVID19_FaceMaskDetection/"

In [3]:
import sys
sys.path.append(gDrivePath)

In [4]:
# setting the matplotlib backend so that the images can be saved in the background
import matplotlib
matplotlib.use("Agg")

In [6]:
# import the packages
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import AveragePooling2D
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report
import config
import matplotlib.pyplot as plt
import numpy as np
import os
import glob

ModuleNotFoundError: No module named 'tensorflow.keras'

In [ ]:
# Defining a method to get the number of files given a path
def getNumberOfFiles(path):
	list1 = []
	for file_name in glob.iglob(path+'/**/*.jpg', recursive=True):
  		list1.append(file_name)
	return len(list1)

In [ ]:
# Defining a method to plot training and validation accuracy and loss
def plot_training(H, N, plotPath):
	# construct a plot that plots and saves the training history
	plt.style.use("ggplot")
	plt.figure()
	plt.plot(np.arange(0, N), H.history["loss"], label="train_loss")
	plt.plot(np.arange(0, N), H.history["val_loss"], label="val_loss")
	plt.plot(np.arange(0, N), H.history["accuracy"], label="train_acc")
	plt.plot(np.arange(0, N), H.history["val_accuracy"], label="val_acc")
	plt.title("Training Loss and Accuracy")
	plt.xlabel("Epoch #")
	plt.ylabel("Loss/Accuracy")
	plt.legend(loc="lower left")
	plt.savefig(plotPath)

In [ ]:
# Defininf the paths to the training, validation, and testing directories
trainPath = os.path.sep.join([config.BASE_PATH, config.TRAIN])
valPath = os.path.sep.join([config.BASE_PATH, config.VAL])
testPath = os.path.sep.join([config.BASE_PATH, config.TEST])

In [ ]:
# Determine the total number of image paths in training, validation and testing directories
totalTrain = getNumberOfFiles(trainPath)
totalVal = getNumberOfFiles(valPath)
totalTest = getNumberOfFiles(testPath)

In [ ]:
# Initialize the training data augmentation object
## preprocess_input will scale input pixels between -1 and 1
## rotation_range is a value in degrees (0-180), a range within which to randomly rotate pictures
## zoom_range is for randomly zooming inside pictures
## width_shift and height_shift are ranges (as a fraction of total width or height) within which to randomly translate pictures vertically or horizontally
## shear_range is for randomly applying shearing transformations
### Shearing Transformation is a transformation in which all points along a given line remain fixed while other points are shifted parallel to by a distance proportional to their perpendicular distance from.
## horizontal_flip is for randomly flipping half of the images horizontally
## fill_mode is the strategy used for filling in newly created pixels, which can appear after a rotation or a width/height shift

trainAug = ImageDataGenerator(
	preprocessing_function=preprocess_input,
	rotation_range=30,
	zoom_range=0.15,
	width_shift_range=0.2,
	height_shift_range=0.2,
	shear_range=0.15,
	horizontal_flip=True,
	fill_mode="nearest")

In [ ]:
# Initialize the validation data augmentation object
valAug = ImageDataGenerator(preprocessing_function=preprocess_input)

In [ ]:
# Initialize the training generator
trainGen = trainAug.flow_from_directory(
	trainPath,
	class_mode="categorical",
	target_size=(224, 224),
	color_mode="rgb",
	shuffle=True,
	batch_size=config.BATCH_SIZE)

In [ ]:
# Initialize the validation generator
valGen = valAug.flow_from_directory(
	valPath,
	class_mode="categorical",
	target_size=(224, 224),
	color_mode="rgb",
	shuffle=False,
	batch_size=config.BATCH_SIZE)

In [ ]:
# Initialize the testing generator
testGen = valAug.flow_from_directory(
	testPath,
	class_mode="categorical",
	target_size=(224, 224),
	color_mode="rgb",
	shuffle=False,
	batch_size=config.BATCH_SIZE)

In [ ]:
# Loading the MobileNetV2, ensuring the head Full Connected layers are left off
baseModel = MobileNetV2(weights="imagenet", include_top=False, input_tensor=Input(shape=(224, 224, 3)))

In [ ]:
# Construct the head of the model that will be placed on top of the the base model
## Flattening is converting the data into a 1-dimensional array for inputting it to the next layer. 
## We flatten the output of the convolutional layers to create a single long feature vector. 
### Average pooling computes the average of the elements present in the region of feature map covered by the filter.
#### ReLU stands for Rectified Linear Unit. 
#### The main advantage of using the ReLU function over other activation functions is that it does not activate all the neurons at the same time.
## Dropout is a technique where randomly selected neurons are ignored during training.

headModel = baseModel.output
headModel = AveragePooling2D(pool_size=(7, 7))(headModel)
headModel = Flatten(name="flatten")(headModel)
headModel = Dense(128, activation="relu")(headModel)
headModel = Dropout(0.5)(headModel)
headModel = Dense(len(config.CLASSES), activation="softmax")(headModel)

In [ ]:
# Placing the head model on top of the base model
model = Model(inputs=baseModel.input, outputs=headModel)

In [ ]:
# Loop over all the layers of the base model and freeze them so that they are 
# not updated during the training process
for layer in baseModel.layers:
	layer.trainable = False

In [ ]:
# Compiling the model
## Learning rate decay is a technique for training modern neural networks. It starts training the network with a large learning rate and then slowly reducing/decaying it until local minima is obtained.
### In short it means, Decay updates the learning rate by a decreasing factor in each epoch.
print("Compiling model")
opt = Adam(lr=config.INIT_LR, decay=config.INIT_LR / config.EPOCHS)
model.compile(loss="binary_crossentropy", optimizer=opt, metrics=["accuracy"])

In [ ]:
# Fitting the model on training data
print("Fitting Model")
H = model.fit(
	x=trainGen,
	steps_per_epoch=totalTrain // config.BATCH_SIZE,
	validation_data=valGen,
	validation_steps=totalVal // config.BATCH_SIZE,
	epochs=config.EPOCHS)

In [ ]:
# Predicting on the test data
print("Predicting on the test data")
predIdxs = model.predict(x=testGen,
	steps=(totalTest // config.BATCH_SIZE) + 1)
predIdxs = np.argmax(predIdxs, axis=1)

In [ ]:
# Printing the Classification Report
print(classification_report(testGen.classes, predIdxs, target_names=testGen.class_indices.keys()))

In [ ]:
# Plotting the graph
plot_training(H, config.EPOCHS, config.PLOT_PATH)

In [ ]:
# Serialize/Writing the model to disk
print("Serializing network...")
model.save(config.MODEL_PATH, save_format="h5")